# Pipeline 5 - Safehouse Capacity and Incident Forecast

Generated: 2026-04-06T22:13:08.304438Z


## CRISP-DM / Rubric Overview

Business understanding: Forecast next-month incident and capacity pressure for safehouse planning.

Data understanding and preparation are implemented in code below. Each notebook includes a predictive model, an explanatory model, evaluation metrics, relationship-analysis notes, and JSON export for app deployment.


In [ ]:
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, precision_recall_fscore_support, r2_score, roc_auc_score, top_k_accuracy_score
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "ml-pipelines" else cwd
DATA_DIR = REPO_ROOT / "data" / "raw"
OUT_DIR = REPO_ROOT / "output" / "ml-predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

def require_csv(stem):
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the INTEX CSVs under data/raw/.")
    return pd.read_csv(path, encoding="utf-8-sig")

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def as_bool(series):
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])

def numeric(series, fill_value=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(fill_value)

def fill_numeric_median(df, cols):
    for col in cols:
        vals = pd.to_numeric(df[col], errors="coerce")
        df[col] = vals.fillna(vals.median() if vals.notna().any() else 0.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None and pd.Series(y_true).nunique() > 1:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out

def eval_regression(y_true, y_pred):
    return {"mae": float(mean_absolute_error(y_true, y_pred)), "rmse": rmse(y_true, y_pred), "r2": float(r2_score(y_true, y_pred))}

def classification_baseline(y_train, y_test):
    majority = pd.Series(y_train).mode().iloc[0]
    pred = pd.Series([majority] * len(y_test), index=pd.Series(y_test).index)
    return {"majority_class": str(majority), "accuracy": float(accuracy_score(y_test, pred))}

def regression_baseline(y_train, y_test):
    median_value = float(pd.Series(y_train).median())
    pred = np.repeat(median_value, len(y_test))
    out = eval_regression(y_test, pred)
    out["baseline_value"] = median_value
    return out

def top_features(pipe, n=10):
    if "pre" not in pipe.named_steps:
        return pd.DataFrame()
    model = pipe.named_steps.get("model") or pipe.named_steps.get("m")
    if model is None:
        return pd.DataFrame()
    try:
        names = pipe.named_steps["pre"].get_feature_names_out()
    except Exception:
        return pd.DataFrame()
    if hasattr(model, "feature_importances_"):
        weights = np.asarray(model.feature_importances_)
    elif hasattr(model, "coef_"):
        weights = np.abs(np.asarray(model.coef_)).mean(axis=0) if np.asarray(model.coef_).ndim > 1 else np.abs(np.asarray(model.coef_))
    else:
        return pd.DataFrame()
    out = pd.DataFrame({"feature": names, "importance": weights}).sort_values("importance", ascending=False).head(n)
    return out.reset_index(drop=True)

def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)

def compact_cv_classification(X, y, cat_cols, num_cols, scoring_metric="roc_auc", cv_splits=3):
    y = pd.Series(y)
    min_class = y.value_counts().min()
    if y.nunique() < 2 or min_class < 2:
        print("Skipping CV comparison: target does not have enough samples in each class.")
        return pd.DataFrame()
    n_splits = int(min(cv_splits, min_class))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    models = {
        "Logistic Regression": LogisticRegression(max_iter=3000),
        "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42, min_samples_leaf=3),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    }
    rows = []
    scoring = {"accuracy": "accuracy", "f1_weighted": "f1_weighted"}
    if y.nunique() == 2:
        scoring["roc_auc"] = "roc_auc"
    for name, model in models.items():
        pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", model)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=1, error_score=np.nan)
        row = {"Model": name}
        for metric in scoring:
            vals = scores[f"test_{metric}"]
            row[f"{metric}_mean"] = float(np.nanmean(vals))
            row[f"{metric}_std"] = float(np.nanstd(vals))
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(f"{'roc_auc' if 'roc_auc' in scoring else 'accuracy'}_mean", ascending=False)
    print("Compact cross-validation comparison:")
    print(out.to_string(index=False))
    return out

def compact_holdout_regression(train, test, features, target, cat_cols, num_cols, log_target=False):
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42, min_samples_leaf=3),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }
    rows = []
    best = None
    best_mae = float("inf")
    y_train = np.log1p(train[target]) if log_target else train[target]
    for name, model in models.items():
        pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", model)])
        pipe.fit(train[features], y_train)
        pred = pipe.predict(test[features])
        pred = np.expm1(pred) if log_target else pred
        pred = np.maximum(0, pred)
        metrics = eval_regression(test[target], pred)
        rows.append({"Model": name, **metrics})
        if metrics["mae"] < best_mae:
            best_mae = metrics["mae"]
            best = (name, pipe)
    out = pd.DataFrame(rows).sort_values("mae")
    print("Time/holdout model comparison:")
    print(out.to_string(index=False))
    return out, best

def compact_randomized_tune_regressor(train, features, target, cat_cols, num_cols, log_target=False, n_iter=6):
    y_train = np.log1p(train[target]) if log_target else train[target]
    cv = KFold(n_splits=min(3, max(2, len(train) // 20)), shuffle=True, random_state=42)
    pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestRegressor(random_state=42))])
    search = RandomizedSearchCV(
        pipe,
        {
            "model__n_estimators": [100, 150, 250],
            "model__min_samples_leaf": [1, 3, 5, 8],
            "model__max_depth": [None, 3, 5, 8],
        },
        n_iter=n_iter,
        scoring="neg_mean_absolute_error",
        cv=cv,
        random_state=42,
        n_jobs=1,
    )
    search.fit(train[features], y_train)
    print("RandomizedSearchCV best params:", search.best_params_)
    print("RandomizedSearchCV best CV MAE:", float(-search.best_score_))
    return search.best_estimator_

def time_split(df, date_col, test_frac=0.25):
    df = df.sort_values(date_col).copy()
    split_idx = max(1, int(len(df) * (1 - test_frac)))
    split_idx = min(split_idx, len(df) - 1)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def safe_classifier_split(X, y, test_size=0.25):
    stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
    return train_test_split(X, y, test_size=test_size, random_state=42, stratify=stratify)

def score_bands(scores):
    scores = pd.Series(scores).astype(float)
    if len(scores) == 0 or scores.nunique(dropna=True) < 2:
        return pd.Series(["Medium"] * len(scores), index=scores.index)
    labels = ["Low", "Medium", "High", "Very High"]
    q = min(4, scores.nunique(dropna=True), len(scores))
    try:
        return pd.qcut(scores.rank(method="first"), q=q, labels=labels[:q], duplicates="drop").astype(str)
    except Exception:
        return pd.Series(["Medium"] * len(scores), index=scores.index)

def prep(cat_cols, num_cols):
    return ColumnTransformer([("cat", make_encoder(), cat_cols), ("num", "passthrough", num_cols)])

def export_predictions_json(prediction_type, entity_type, df_out, id_col, score_col, label_col=None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    excluded = {id_col, score_col}
    if label_col:
        excluded.add(label_col)
    rows = []
    for _, row in df_out.iterrows():
        payload = {k: v for k, v in row.items() if k not in excluded}
        rows.append({
            "predictionType": prediction_type,
            "entityType": entity_type,
            "entityId": int(row[id_col]),
            "score": float(row[score_col]),
            "label": None if label_col is None or pd.isna(row[label_col]) else str(row[label_col]),
            "payloadJson": json.dumps(payload, default=str),
        })
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print(f"Wrote {len(rows)} predictions:", out_path)


In [ ]:
metrics = require_csv("safehouse_monthly_metrics")
metrics["month_start"] = pd.to_datetime(metrics["month_start"], errors="coerce")
metrics = metrics.dropna(subset=["month_start"]).sort_values(["safehouse_id","month_start"]).copy()
num_base = ["active_residents","avg_education_progress","avg_health_score","process_recording_count","home_visitation_count","incident_count"]
fill_numeric_median(metrics, num_base)
g = metrics.groupby("safehouse_id")
metrics["incident_lag1"] = g["incident_count"].shift(1)
metrics["incident_lag2"] = g["incident_count"].shift(2)
metrics["active_lag1"] = g["active_residents"].shift(1)
metrics["active_lag2"] = g["active_residents"].shift(2)
metrics["rolling_incident_3m"] = g["incident_count"].transform(lambda s: s.rolling(3, min_periods=1).mean())
metrics["incident_next"] = g["incident_count"].shift(-1)
metrics["active_residents_next"] = g["active_residents"].shift(-1)
df = metrics.dropna(subset=["incident_next","active_residents_next","incident_lag1","active_lag1"]).copy()
features = num_base + ["incident_lag1","incident_lag2","active_lag1","active_lag2","rolling_incident_3m"]
fill_numeric_median(df, features + ["incident_next","active_residents_next"])
train, test = time_split(df, "month_start")
print("Rows:", len(df), "Train:", len(train), "Test:", len(test))


In [ ]:
incident_model = RandomForestRegressor(n_estimators=300, random_state=42)
print("Incident baseline:", regression_baseline(train["incident_next"], test["incident_next"]))
compact_holdout_regression(train, test, features, "incident_next", [], features, log_target=False)
incident_model.fit(train[features], train["incident_next"])
incident_pred = np.maximum(0, incident_model.predict(test[features]))
print("Predictive incident forecast:", eval_regression(test["incident_next"], incident_pred))
print("Top incident forecast features:")
print(pd.DataFrame({"feature": features, "importance": incident_model.feature_importances_}).sort_values("importance", ascending=False).head(10).to_string(index=False))

active_model = RandomForestRegressor(n_estimators=300, random_state=42)
print("Capacity baseline:", regression_baseline(train["active_residents_next"], test["active_residents_next"]))
compact_holdout_regression(train, test, features, "active_residents_next", [], features, log_target=False)
active_model.fit(train[features], train["active_residents_next"])
active_pred = np.maximum(0, active_model.predict(test[features]))
print("Predictive capacity forecast:", eval_regression(test["active_residents_next"], active_pred))

explanatory = LinearRegression()
explanatory.fit(train[features], train["incident_next"])
incident_exp = np.maximum(0, explanatory.predict(test[features]))
print("Explanatory incident model:", eval_regression(test["incident_next"], incident_exp))
print_business_takeaway("Use the safehouse forecast as a relative ranking for planning staffing and attention, not as an exact count forecast.")


In [ ]:
latest = metrics.groupby("safehouse_id").tail(1).copy()
fill_numeric_median(latest, features)
latest["predicted_incidents_next_month"] = np.maximum(0, incident_model.predict(latest[features]))
latest["predicted_active_residents_next_month"] = np.maximum(0, active_model.predict(latest[features]))
latest["risk_band"] = score_bands(latest["predicted_incidents_next_month"])
export_predictions_json(
    "safehouse_incident_next_month",
    "Safehouse",
    latest[["safehouse_id","predicted_incidents_next_month","risk_band","predicted_active_residents_next_month","active_residents","incident_count","avg_health_score","avg_education_progress"]],
    "safehouse_id",
    "predicted_incidents_next_month",
    "risk_band",
)


## Deployment

Import the exported JSON with `POST /api/ml/import?replace=true` and view results in `/app/ml`.
